<a href="https://colab.research.google.com/github/niranjanniru-max/House-Price-Prediction-ipynb-file/blob/main/House_Price_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import kagglehub
import pandas as pd
import os

# The name of the specific file within the dataset you want to load
file_name = "data.csv"

# Download the dataset. This caches the dataset locally and returns the path to its directory.
dataset_dir = kagglehub.dataset_download("bhanupratapbiswas/house-price-prediction")

# Construct the full path to the specific file
file_path = os.path.join(dataset_dir, file_name)

# Load the dataset into a pandas DataFrame and store it as raw_data
raw_data = pd.read_csv(file_path)

# For initial display or if 'data' is expected immediately after loading
data = raw_data.copy()

Using Colab cache for faster access to the 'house-price-prediction' dataset.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

from xgboost import XGBRegressor

from sklearn.metrics import r2_score, mean_squared_error

In [ ]:
data = data.drop_duplicates()

data = data[data['price'] > 0]

data['date'] = pd.to_datetime(data['date'])

data['sale_year'] = data['date'].dt.year
data['sale_month'] = data['date'].dt.month

data['house_age'] = (
    data['sale_year']
    - data['yr_built']
)

data['is_renovated'] = (
    data['yr_renovated'] > 0
).astype(int)

data['total_sqft'] = (
    data['sqft_above']
    + data['sqft_basement']
)

data.drop(
    [
        'date',
        'country'
    ],
    axis=1,
    inplace=True,
    errors='ignore'
)

In [ ]:
X = data.drop('price', axis=1)

y = np.log1p(data['price'])

cat_cols = X.select_dtypes(
    include='object'
).columns

num_cols = X.select_dtypes(
    exclude='object'
).columns

In [ ]:
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, num_cols),
    ('cat', categorical_transformer, cat_cols)
])


In [ ]:

model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', model)
])

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

pipe.fit(X_train, y_train)

pred_log = pipe.predict(X_test)

pred = np.expm1(pred_log)
actual = np.expm1(y_test)

In [ ]:
print("R2:",r2_score(actual, pred))

print("MSE:",mean_squared_error(actual, pred))

result = pd.DataFrame({"Actual": actual,"Predicted": pred})

pd.options.display.float_format = '{:,.0f}'.format
result = pd.DataFrame({"Actual": actual, "Predicted": pred})
print(result.head(10))


R2: 0.7255035612657774
MSE: 40836942433.20174
        Actual  Predicted
471  1,225,000  1,243,631
2518   496,752    485,282
23     612,500    597,586
3922   265,000    259,098
135    615,000    644,612
1789   432,000    413,500
3335   305,000    350,751
2374   405,000    325,637
2320   349,000    465,202
3260   839,000    813,913


I implemented the House Price Prediction task myself, and my version goes beyond the one in the PDF. Their version mainly used LabelEncoder/MinMaxScaler, separate steps, and focused on a single model with visualizations. Mine is different: I built a proper ML pipeline with a ColumnTransformer (StandardScaler + OneHotEncoder), compared multiple models (Logistic Regression, Random Forest, Decision Tree), and tuned hyperparameters using GridSearchCV. I reported CV score, best params, classification report, and test accuracy — so it’s structured, reusable, and closer to how ML is done in production.

The major difference is that their version is more exploratory with plots, while mine is engineering‑oriented with pipelines and metrics. I focused on numbers because that’s how I understand results best, even though I’m still improving at visualization. I’ve studied ML thoroughly during this internship, and this project reflects that learning.